# Tutorial 05: Autoencoders e Detecção Não-Supervisionada de Anomalias no FPGA

Neste tutorial, exploramos a utilização de **Autoencoders** implementados em FPGA para detecção de anomalias em tempo real com `hls4ml`.

### Contexto Físico e Aplicação em Tempo Real:
- Em experimentos científicos de alta taxa (como o **LHC no CERN**) ou em monitoramento industrial de alta velocidade, ocorrem milhões de eventos por segundo;
- A grande maioria dos eventos consiste em processos comuns conhecidos (**Background**, como jatos QCD de glúons e quarks leves);
- Processos raros ou novas partículas (**Sinal Anômalo**, como quarks Top, bósons W/Z ou física além do Modelo Padrão) são extremamente infrequentes;
- Classificadores supervisionados só encontram anomalias que já conhecem. Um **Autoencoder**, treinado *apenas* com o fundo comum, aprende a representação do que é "normal";
- Quando um evento anômalo entra no pipeline, o hardware falha em reconstruí-lo fielmente, gerando um **Erro Médio Quadrático (MSE)** elevado que aciona o gatilho de seleção (*trigger*).

## 1. Importações e Configuração do Ambiente

In [ ]:
import os
import sys
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from models import JetAutoencoder
import hls4ml

print(f"hls4ml : {hls4ml.__version__}")
print(f"PyTorch: {torch.__version__}")

## 2. Preparação dos Dados: Separação do Fundo (Background QCD)
Selecionamos apenas os jatos de glúons (`g`) e quarks leves (`q`) para o treinamento não-supervisionado do Autoencoder.

In [ ]:
data_dir = os.path.join(ROOT_DIR, "data", "jet-tagging")
X_train = np.load(os.path.join(data_dir, "X_train.npy"))
y_train = np.load(os.path.join(data_dir, "y_train.npy"))
X_test = np.load(os.path.join(data_dir, "X_test.npy"))
y_test = np.load(os.path.join(data_dir, "y_test.npy"))
classes = list(np.load(os.path.join(data_dir, "classes.npy"), allow_pickle=True))

idx_g, idx_q, idx_t = classes.index("g"), classes.index("q"), classes.index("t")
is_bkg_train = (y_train[:, idx_g] == 1) | (y_train[:, idx_q] == 1)
X_train_bkg = X_train[is_bkg_train][:100000].astype(np.float32)

print(f"Amostras de Background para treino não-supervisionado: {len(X_train_bkg)}")

## 3. Modelo PyTorch (JetAutoencoder)
Topologia com gargalo latente (*bottleneck*):
- **Encoder**: `Linear(16 -> 8) -> ReLU -> Linear(8 -> 3) -> ReLU` (comprime 16 variáveis físicas em apenas 3 variáveis latentes);
- **Decoder**: `Linear(3 -> 8) -> ReLU -> Linear(8 -> 16)` (reconstrói as 16 variáveis).

In [ ]:
device = torch.device("cpu")
model = JetAutoencoder().to(device)

weights_path = os.path.join(ROOT_DIR, "models", "pytorch_weights_autoencoder.pt")
if os.path.exists(weights_path):
    print(f"Carregando pesos pré-treinados de {weights_path}...")
    model.load_state_dict(torch.load(weights_path, map_location=device))
else:
    print("Treinando Autoencoder...")
    train_ds = TensorDataset(torch.from_numpy(X_train_bkg), torch.from_numpy(X_train_bkg))
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    for epoch in range(1, 6):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        print(f"Época {epoch}/5 concluída.")
    torch.save(model.state_dict(), weights_path)

# Avaliação do Erro Médio Quadrático (MSE) no conjunto de teste
eval_n = 10000
X_eval = X_test[:eval_n].astype(np.float32)
y_eval = y_test[:eval_n]

model.eval()
with torch.no_grad():
    recon_pt = model(torch.from_numpy(X_eval)).cpu().numpy()
mse_pt = np.mean((X_eval - recon_pt) ** 2, axis=1)

is_bkg_eval = (y_eval[:, idx_g] == 1) | (y_eval[:, idx_q] == 1)
is_anomaly_eval = (y_eval[:, idx_t] == 1)
mask = is_bkg_eval | is_anomaly_eval

fpr_pt, tpr_pt, _ = roc_curve(is_anomaly_eval[mask], mse_pt[mask])
auc_pt = auc(fpr_pt, tpr_pt)
print(f"AUC de Detecção de Anomalia (PyTorch Float32): {auc_pt:.4f}")

## 4. Conversão para Firmware FPGA com hls4ml
Configuramos a precisão padrão como `ap_fixed<16,6>` com estratégia de latência mínima (`io_parallel`, `ReuseFactor = 1`).

In [ ]:
config = hls4ml.utils.config_from_pytorch_model(model, input_shape=(16,))
config["Model"]["Precision"]["default"] = "ap_fixed<16,6>"
config["Model"]["Strategy"] = "Latency"
config["Model"]["ReuseFactor"] = 1

output_dir = "hls4ml_prj_autoencoder"
hls_model = hls4ml.converters.convert_from_pytorch_model(
    model,
    input_shape=(16,),
    hls_config=config,
    output_dir=output_dir,
    part="xc7z020clg400-1",
    io_type="io_parallel"
)
print("Projeto de hardware sintetizado com sucesso!")

## 5. Compilação e C-Simulation Nativa (`g++`)

In [ ]:
t0 = time.time()
hls_model.compile()
print(f"Compilação g++ finalizada em {time.time() - t0:.2f}s.")

t1 = time.time()
recon_hls = hls_model.predict(X_eval)
print(f"C-Simulation em {eval_n} eventos concluída em {time.time() - t1:.2f}s.")

mse_hls = np.mean((X_eval - recon_hls) ** 2, axis=1)
fpr_hls, tpr_hls, _ = roc_curve(is_anomaly_eval[mask], mse_hls[mask])
auc_hls = auc(fpr_hls, tpr_hls)

print(f"AUC PyTorch (Float32)         : {auc_pt:.4f}")
print(f"AUC hls4ml (ap_fixed<16,6>)   : {auc_hls:.4f}")
print(f"Diferença Absoluta de AUC     : {abs(auc_pt - auc_hls):.4f}")

## 6. Visualização: Distribuição de Erro e Curva ROC de Anomalia

In [ ]:
# Histograma da perda de reconstrução
plt.figure(figsize=(9, 4.5))
bins = np.linspace(0, 0.4, 60)
plt.hist(mse_hls[is_bkg_eval], bins=bins, alpha=0.6, color="#1f77b4", label="Fundo Conhecido (Gluon + Quark)", density=True)
plt.hist(mse_hls[is_anomaly_eval], bins=bins, alpha=0.6, color="#d62728", label="Anomalia (Top Quarks)", density=True)
plt.xlabel("Erro Médio Quadrático de Reconstrução (MSE)", fontsize=11)
plt.ylabel("Densidade de Probabilidade", fontsize=11)
plt.title("Distribuição do Erro de Reconstrução no FPGA (hls4ml ap_fixed<16,6>)", fontsize=12, fontweight="bold")
plt.legend(loc="upper right", fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

# Curva ROC
plt.figure(figsize=(7.5, 5.5))
plt.plot(fpr_pt, tpr_pt, "b--", lw=1.8, label=f"PyTorch Float32 (AUC = {auc_pt:.3f})")
plt.plot(fpr_hls, tpr_hls, "r-", lw=1.8, label=f"hls4ml ap_fixed<16,6> (AUC = {auc_hls:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Baseline Aleatório (AUC = 0.500)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("Taxa de Falso Positivo (FPR)", fontsize=11)
plt.ylabel("Taxa de Verdadeiro Positivo (TPR)", fontsize=11)
plt.title("Detecção Não-Supervisionada de Anomalias: Curva ROC", fontsize=12, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()